# Stage 4 — Retrain (fast: pre-cached crops)
Materialises crops to disk ONCE, then trains from them with ImageFolder
(fast) instead of decoding full frames every epoch (slow).

In [1]:
import sys; sys.path.insert(0,'/workspace/lib')
import os, json, time, random
import numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import dataset_lib as D
from dataset_lib import CLASSES, pool_paths

In [2]:
# --- config ---
DATASET_ROOT = '/dataset'
MANIFEST_VERSION = D.latest_manifest_version(DATASET_ROOT)
DEEPFAUNE_CKPT = '/workspace/deepfaune-vit_large_patch14_dinov2.lvd142m.v4.pt'
MODEL_VERSION = MANIFEST_VERSION
SEED = 42
IMG_SIZE=518; BATCH=128; P1_EPOCHS=5; P2_EPOCHS=0; P1_LR=1e-3; P2_LR=1e-5
WEIGHT_DECAY=0.05; LABEL_SMOOTH=0.1
MEAN=(0.485,0.456,0.406); STD=(0.229,0.224,0.225)
PAD=0.12
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device='cuda' if torch.cuda.is_available() else 'cpu'
print('Training model', MODEL_VERSION, 'on', device)
man = D.load_manifest(DATASET_ROOT, MANIFEST_VERSION)
print('images:', man['n_images'], '| splits:', man['split_counts'])
CROPS_ROOT = f'/dataset/crops/{MANIFEST_VERSION}'

Training model v1.1 on cuda
images: 48061 | splits: {'train': 38413, 'val': 4799, 'test': 4849}


## 1. Materialise crops to disk (run ONCE per version)
If `/dataset/crops/<version>/` already exists from a previous run, you can
skip this cell and go straight to the dataloader.

In [3]:
# crop every labelled box once -> /dataset/crops/<version>/<split>/<Class>/
img_dir, lbl_dir = pool_paths(DATASET_ROOT)
id_to_path = {os.path.splitext(f)[0]: os.path.join(img_dir,f) for f in os.listdir(img_dir)}

def materialise(split):
    n_ok=n_bad=0
    for e in man['images']:
        if e['split']!=split: continue
        iid=e['id']; ip=id_to_path.get(iid); lp=os.path.join(lbl_dir, iid+'.txt')
        if ip is None or not os.path.exists(lp): continue
        try:
            im=Image.open(ip).convert('RGB'); W,H=im.size
        except Exception:
            n_bad+=1; continue
        for bi,line in enumerate(open(lp)):
            p=line.split()
            if len(p)<5: continue
            cid=int(float(p[0])); cx,cy,bw,bh=map(float,p[1:5])
            x1=max(0,int((cx-bw/2-bw*PAD)*W)); y1=max(0,int((cy-bh/2-bh*PAD)*H))
            x2=min(W,int((cx+bw/2+bw*PAD)*W)); y2=min(H,int((cy+bh/2+bh*PAD)*H))
            if x2<=x1 or y2<=y1: x1,y1,x2,y2=0,0,W,H
            out_dir=os.path.join(CROPS_ROOT,split,CLASSES[cid]); os.makedirs(out_dir,exist_ok=True)
            im.crop((x1,y1,x2,y2)).save(os.path.join(out_dir,f'{iid}_{bi}.jpg'),quality=92)
            n_ok+=1
    return n_ok,n_bad

for split in ['train','val','test']:
    t=time.time(); ok,bad=materialise(split)
    print(f'{split}: {ok} crops ({bad} unreadable skipped) in {time.time()-t:.0f}s')
print('cached under', CROPS_ROOT)

train: 45894 crops (0 unreadable skipped) in 2009s
val: 5703 crops (0 unreadable skipped) in 399s
test: 5766 crops (0 unreadable skipped) in 380s
cached under /dataset/crops/v1.1


## 2. Fast dataloader (ImageFolder over cached crops)

In [4]:
class ResizePadSquare:
    def __init__(s,sz): s.sz=sz
    def __call__(s,im):
        w,h=im.size; sc=s.sz/max(w,h); nw,nh=round(w*sc),round(h*sc)
        im=im.resize((nw,nh),Image.BILINEAR); c=Image.new('RGB',(s.sz,s.sz),(0,0,0))
        c.paste(im,((s.sz-nw)//2,(s.sz-nh)//2)); return c

train_tf=transforms.Compose([ResizePadSquare(IMG_SIZE),transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2,0.05),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
eval_tf=transforms.Compose([ResizePadSquare(IMG_SIZE),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

name_to_canon={name:i for i,name in enumerate(CLASSES)}
def make(split,tf):
    ds=datasets.ImageFolder(f'{CROPS_ROOT}/{split}',transform=tf)
    idx_map={i:name_to_canon[name] for i,name in enumerate(ds.classes)}
    ds.samples=[(p,idx_map[t]) for (p,t) in ds.samples]; ds.targets=[idx_map[t] for t in ds.targets]
    return ds

train_ds=make('train',train_tf); val_ds=make('val',eval_tf); test_ds=make('test',eval_tf)
g=torch.Generator(); g.manual_seed(SEED)
train_dl=DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=8,pin_memory=True,drop_last=True,generator=g,persistent_workers=True,prefetch_factor=4)
val_dl=DataLoader(val_ds,batch_size=BATCH,num_workers=8,pin_memory=True,persistent_workers=True,prefetch_factor=4)
test_dl=DataLoader(test_ds,batch_size=BATCH,num_workers=8,pin_memory=True,persistent_workers=True,prefetch_factor=4)
print('crops:',len(train_ds),'train',len(val_ds),'val',len(test_ds),'test')
print('batch in use:',train_dl.batch_size,'| batches/epoch:',len(train_dl))

crops: 45894 train 5703 val 5766 test
batch in use: 128 | batches/epoch: 358


## 3. Model — DeepFaune backbone + fresh 30-class head

In [5]:
model=timm.create_model('vit_large_patch14_dinov2.lvd142m',pretrained=False,num_classes=len(CLASSES),img_size=IMG_SIZE)
raw=torch.load(DEEPFAUNE_CKPT,map_location='cpu',weights_only=False)
sd=raw['state_dict'] if isinstance(raw,dict) and 'state_dict' in raw else raw
sd={k.replace('base_model.','').replace('module.',''):v for k,v in sd.items()}
for hk in ['head.weight','head.bias']: sd.pop(hk,None)
res=model.load_state_dict(sd,strict=False)
assert all('head' in m for m in res.missing_keys), res.missing_keys
model.to(device); print('backbone loaded; head reinit for',len(CLASSES),'classes')

backbone loaded; head reinit for 30 classes


## 4. Train

In [6]:
crit=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
def run(dl,train,opt=None):
    model.train(train); tot=cor=0; ls=0.0
    for x,y in dl:
        x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
        with torch.set_grad_enabled(train):
            with torch.autocast('cuda',dtype=torch.bfloat16,enabled=(device=='cuda')):
                o=model(x); l=crit(o,y)
            if train: opt.zero_grad(set_to_none=True); l.backward(); opt.step()
        ls+=l.item()*x.size(0); cor+=(o.argmax(1)==y).sum().item(); tot+=x.size(0)
    return ls/tot, cor/tot
def freeze(f):
    for n,p in model.named_parameters():
        if 'head' not in n: p.requires_grad=not f
MODEL_DIR=f'/dataset/models/{MODEL_VERSION}'; os.makedirs(MODEL_DIR,exist_ok=True)

In [7]:
# Phase 1: linear probe
freeze(True); opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=P1_LR,weight_decay=WEIGHT_DECAY)
best=0
for e in range(P1_EPOCHS):
    t=time.time(); _,ta=run(train_dl,True,opt); _,va=run(val_dl,False)
    print(f'[P1 {e+1}/{P1_EPOCHS}] train {ta:.3f} val {va:.3f} {time.time()-t:.0f}s')
    if va>best: best=va; torch.save(model.state_dict(),f'{MODEL_DIR}/best.pt')
print('P1 best',best)

[P1 1/5] train 0.974 val 0.994 750s
[P1 2/5] train 0.995 val 0.994 730s
[P1 3/5] train 0.997 val 0.995 728s
[P1 4/5] train 0.997 val 0.996 727s
[P1 5/5] train 0.997 val 0.996 731s
P1 best 0.9963177275118359


In [8]:
# Phase 2: full fine-tune (P2_EPOCHS=0 skips this — probe only, like v1.0)
if P2_EPOCHS>0:
    freeze(False); opt=torch.optim.AdamW(model.parameters(),lr=P2_LR,weight_decay=WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=P2_EPOCHS)
    for e in range(P2_EPOCHS):
        t=time.time(); _,ta=run(train_dl,True,opt); _,va=run(val_dl,False); sch.step()
        print(f'[P2 {e+1}/{P2_EPOCHS}] train {ta:.3f} val {va:.3f} {time.time()-t:.0f}s')
        if va>best: best=va; torch.save(model.state_dict(),f'{MODEL_DIR}/best.pt'); print('  saved')
    print('best val',best)
else:
    print('Phase 2 skipped (P2_EPOCHS=0)')

Phase 2 skipped (P2_EPOCHS=0)


Weights in `/dataset/models/<version>/best.pt`. Run **Stage 5** to evaluate + export ONNX.